# Pattern B · Cross App Access (ID-JAG) with a user

The same chain as Pattern A, but **no authorization server trusts another**. Your backend and the agent are each a registered **Okta AI agent**, and each hop is two grants: a token exchange at Okta's **org** server for an **ID-JAG**, then a **JWT bearer** grant at the target server.

| Step | Who asks | Okta app / AI agent | Gets | From | Grant |
|---|---|---|---|---|---|
| 1 | Ada (sign-in) | `anyco-eng-login-app` | T1 | `anyco-users-as` | authorization code + PKCE |
| 2a | your backend | `anyco-backend-agent-oai` | ID-JAG | Okta org server | token exchange, subject T1 |
| 2b | your backend | `anyco-backend-agent-oai` | T2 | `anyco-agent-b-as` (trusts nobody) | JWT bearer, assertion = the ID-JAG |
| 3a | the agent | `anyco-pr-review-agent-b-oai` | ID-JAG | Okta org server | token exchange, subject T2 |
| 3b | the agent | `anyco-pr-review-agent-b-oai` | T3 | `anyco-gateway-as` | JWT bearer, assertion = the ID-JAG |

Every client assertion is a `private_key_jwt` signed by an AWS KMS key: AgentCore Identity asks KMS to sign with **your** AWS credentials, which the keys' policies allow for the console operator (`kms_keys.py`).

**Before you run it:** Okta is set up (part 3 of `docs/okta-setup.md`), and `okta_values.py` and `deploy.py` have run (they create the agent's providers, `anyco-xaa-step1` and `anyco-xaa-step2`). This notebook creates a workload identity and two providers for the backend; the last cell removes them.

In [ ]:
import asyncio, base64, hashlib, json, os, secrets, sys, time, urllib.parse, uuid
sys.path.insert(0, "..")                       # the repo root: config.py reads ../.env and ../state.json
import requests
import config as C

os.environ["AWS_REGION"] = C.REGION            # the decorator builds its own client from AWS_REGION
if C.AWS_PROFILE:
    os.environ["AWS_PROFILE"] = C.AWS_PROFILE
from bedrock_agentcore.identity.auth import requires_access_token
from bedrock_agentcore.runtime.context import BedrockAgentCoreContext
from bedrock_agentcore.services.identity import IdentityClient

E, st = C.os.environ, C.state()
ac = C.session().client("bedrock-agentcore-control")   # to create the notebook's providers
identity = IdentityClient(C.REGION)                     # to bind a JWT to the workload identity

# An older boto3 doesn't know the KMS-signed client assertion (privateKeyJwtConfig) the providers use.
_fields = ac.meta.service_model.shape_for("CustomOauth2ProviderConfigInput").members
if "privateKeyJwtConfig" not in _fields:
    import boto3
    raise RuntimeError(f"This kernel's boto3 {boto3.__version__} is too old ({sys.executable}). "
                       "Run the notebook on the repo's .venv kernel, or: pip install -r notebooks/requirements.txt")

def claims(jwt):
    return json.loads(base64.urlsafe_b64decode(jwt.split(".")[1] + "=="))

def show(name, jwt):
    """A bearer token: the raw value shortened, then the header and every claim. The signature is shown
    only as a length, so nothing printed here is a usable credential."""
    h, p, sig = jwt.split(".")
    dec = lambda x: json.loads(base64.urlsafe_b64decode(x + "=="))
    print(f"{name}  ·  {len(jwt)} chars  ·  {jwt[:16]}…{jwt[-6:]}")
    print(json.dumps({"header": dec(h), "payload": dec(p), "signature": f"<{len(sig)} chars, not shown>"}, indent=1))

def act_chain(jwt):
    """Every actor in the token, newest first."""
    out, a = [], claims(jwt).get("act")
    while isinstance(a, dict):
        out.append(f"{a['sub']} ({a.get('sub_profile', '')})"); a = a.get("act")
    return out

WORKLOAD = "anyco-notebook-workload"           # a standalone workload identity, to bind each subject token
try:
    ac.create_workload_identity(name=WORKLOAD, tags={"project": "anyco-agentid"})
except ac.exceptions.ValidationException as e:
    if "already exists" not in str(e): raise

def bind(jwt):
    """Make this JWT the subject of the next decorated call: AgentCore Identity returns a workload access
    token bound to it, and the decorators read the one in the context."""
    wat = identity.get_workload_access_token(WORKLOAD, user_token=jwt)["workloadAccessToken"]
    BedrockAgentCoreContext.set_workload_access_token(wat)

def provider(name, cfg):
    """Create or update an AgentCore Identity OAuth2 provider."""
    try:
        ac.create_oauth2_credential_provider(name=name, credentialProviderVendor="CustomOauth2",
                                             oauth2ProviderConfigInput=cfg, tags={"project": "anyco-agentid"})
    except (ac.exceptions.ConflictException, ac.exceptions.ValidationException) as e:
        if "already exists" not in str(e) and not isinstance(e, ac.exceptions.ConflictException): raise
        ac.update_oauth2_credential_provider(name=name, credentialProviderVendor="CustomOauth2", oauth2ProviderConfigInput=cfg)

ACCESS_TOKEN = "urn:ietf:params:oauth:token-type:access_token"
ID_JAG = "urn:ietf:params:oauth:token-type:id-jag"
print("ready ·", C.OKTA_DOMAIN, "·", C.REGION)

## 1 · Ada signs in: T1

In [ ]:
# Ada signs in to anyco-eng-login-app at anyco-users-as: authorization code + PKCE, with plain requests.
# (A browser would show Okta's sign-in page; here Okta's authn API gives a session token instead.)
ok = st["okta"]
s = requests.Session()
authn = s.post(f"{C.OKTA_DOMAIN}/api/v1/authn", timeout=20,
               json={"username": "ada.lovelace@example.com", "password": E["DEMO_USER_PASSWORD"]}).json()
verifier = secrets.token_urlsafe(48)
challenge = base64.urlsafe_b64encode(hashlib.sha256(verifier.encode()).digest()).rstrip(b"=").decode()
r = s.get(f"{ok['users_issuer']}/v1/authorize", allow_redirects=False, timeout=20, params={
    "client_id": ok["login_client_id"], "response_type": "code", "scope": "openid profile email pr-agent.invoke",
    "redirect_uri": C.REDIRECT_URI, "state": uuid.uuid4().hex, "nonce": uuid.uuid4().hex,
    "code_challenge": challenge, "code_challenge_method": "S256", "sessionToken": authn["sessionToken"]})
code = urllib.parse.parse_qs(urllib.parse.urlparse(r.headers["Location"]).query)["code"][0]
tokens = s.post(f"{ok['users_issuer']}/v1/token", auth=(ok["login_client_id"], E["ANYCO_LOGIN_CLIENT_SECRET"]), timeout=20,
                data={"grant_type": "authorization_code", "code": code, "redirect_uri": C.REDIRECT_URI,
                      "code_verifier": verifier}).json()
t1 = tokens["access_token"]
show("T1 · Ada's sign-in token", t1)

## 2 · Your backend: T1 → ID-JAG → T2
`anyco-agent-b-as` trusts no other server, so the backend goes through Okta: the org server issues an ID-JAG addressed to `anyco-agent-b-as` (machine access lets T1 be the subject), and `anyco-agent-b-as` redeems it.

In [ ]:
def xaa_provider(name, discovery_url, grant, client_id, kms_key_arn, kid):
    """A provider that signs its client assertion (private_key_jwt) with an AWS KMS key: no client secret."""
    obo = {"grantType": grant}
    if grant == "TOKEN_EXCHANGE":
        obo["tokenExchangeGrantTypeConfig"] = {"actorTokenContent": "NONE"}
    provider(name, {"customOauth2ProviderConfig": {
        "oauthDiscovery": {"discoveryUrl": discovery_url}, "clientId": client_id,
        "clientAuthenticationMethod": "PRIVATE_KEY_JWT",
        "privateKeyJwtConfig": {"privateKeySource": {"kmsKeySource": {"kmsKeyArn": kms_key_arn}},
                                "signingAlgorithm": "RS256", "additionalHeaderClaims": {"kid": kid}},
        "onBehalfOfTokenExchangeConfig": obo}})

xa = st["xaa"]
STEP1_B, STEP2_B = "anyco-notebook-backend-xaa-step1", "anyco-notebook-backend-xaa-step2"
xaa_provider(STEP1_B, f"{C.OKTA_DOMAIN}/.well-known/openid-configuration", "TOKEN_EXCHANGE",
             xa["backend_agent_client_id"], xa["backend_agent_kms_key_arn"], xa["backend_agent_kid"])
xaa_provider(STEP2_B, f"{xa['agent_b_issuer']}/.well-known/openid-configuration", "JWT_AUTHORIZATION_GRANT",
             xa["backend_agent_client_id"], xa["backend_agent_kms_key_arn"], xa["backend_agent_kid"])

@requires_access_token(provider_name=STEP1_B, scopes=["agent-b.invoke"], auth_flow="ON_BEHALF_OF_TOKEN_EXCHANGE",
                       force_authentication=True,
                       custom_parameters={"subject_token_type": ACCESS_TOKEN, "requested_token_type": ID_JAG,
                                          "audience": xa["agent_b_issuer"]})
async def backend_id_jag(*, access_token):
    return access_token

@requires_access_token(provider_name=STEP2_B, scopes=["agent-b.invoke"], auth_flow="ON_BEHALF_OF_TOKEN_EXCHANGE",
                       force_authentication=True)
async def backend_t2(*, access_token):
    return access_token

bind(t1);           id_jag_1 = await backend_id_jag()      # step 2a: subject T1
show("ID-JAG · for anyco-agent-b-as", id_jag_1)
bind(id_jag_1);     t2 = await backend_t2()                # step 2b: the ID-JAG is the assertion
show("T2 · for the agent", t2)

## 3 · The agent: T2 → ID-JAG → T3
The agent's two providers, created by `deploy.py`, are the ones it uses on AgentCore Runtime. `anyco-gateway-as` doesn't trust `anyco-agent-b-as` either.

In [ ]:
SCOPES = ["pr:read", "pr:review"]

@requires_access_token(provider_name="anyco-xaa-step1", scopes=SCOPES, auth_flow="ON_BEHALF_OF_TOKEN_EXCHANGE",
                       force_authentication=True,
                       custom_parameters={"subject_token_type": ACCESS_TOKEN, "requested_token_type": ID_JAG,
                                          "audience": st["okta"]["gateway_issuer"]})
async def agent_id_jag(*, access_token):
    return access_token

@requires_access_token(provider_name="anyco-xaa-step2", scopes=SCOPES, auth_flow="ON_BEHALF_OF_TOKEN_EXCHANGE",
                       force_authentication=True)
async def agent_t3(*, access_token):
    return access_token

bind(t2);           id_jag_2 = await agent_id_jag()        # step 3a: subject T2
show("ID-JAG · for anyco-gateway-as", id_jag_2)
bind(id_jag_2);     t3 = await agent_t3()                  # step 3b
show("T3 · for the Gateway", t3)

## 4 · The trail
T3 names Ada as `sub`, and every actor: the agent, your backend's AI agent, the sign-in app.

In [ ]:
for name, tok in (("T1", t1), ("T2", t2), ("T3", t3)):
    c = claims(tok)
    print(f"{name}: aud {c['aud']} · sub {c['sub']} · cid {c['cid']}")
    for i, a in enumerate(act_chain(tok)):
        print(f"      {'act' + '.act' * i}.sub = {a}")

## 5 · No trust between servers
A plain token exchange of T1 at `anyco-agent-b-as` is refused: it trusts no other authorization server.

In [ ]:
r = requests.post(f"{xa['agent_b_issuer']}/v1/token", auth=(st["okta"]["backend_client_id"], E["ANYCO_BACKEND_CLIENT_SECRET"]), timeout=20,
                  data={"grant_type": "urn:ietf:params:oauth:grant-type:token-exchange", "subject_token": t1,
                        "subject_token_type": ACCESS_TOKEN, "audience": xa["agent_b_audience"], "scope": "agent-b.invoke"})
print(r.status_code, r.json().get("error"), "·", r.json().get("error_description"))

## Clean up
Removes the workload identity and the two providers this notebook created. The deployed demo is untouched.

In [ ]:
NOTEBOOK_PROVIDERS = ["anyco-notebook-backend-xaa-step1", "anyco-notebook-backend-xaa-step2"]

for name in NOTEBOOK_PROVIDERS:
    try:
        ac.delete_oauth2_credential_provider(name=name); print("deleted provider", name)
    except Exception as e:
        print("skip", name, str(e)[:80])
try:
    ac.delete_workload_identity(name=WORKLOAD); print("deleted workload identity", WORKLOAD)
except Exception as e:
    print("skip", WORKLOAD, str(e)[:80])